# Evaluate AutoGen Group Chat Quality

A group chat is easy to run and hard to read: several agents take turns, one of
them calls a tool, and all you get back is a transcript. This notebook wraps an
[AutoGen (AG2)](https://github.com/ag2ai/ag2) group chat with `TruAutoGen` so each
turn is recorded on its own, then scores the agents separately.

We build a two-agent research team — a researcher with a search tool and a writer —
run it under a `GroupChatManager`, and evaluate:

- **Answer relevance** of the conversation as a whole
- **Coherence of every agent reply**, scored per agent turn rather than averaged
  into the final answer

**Prerequisites:** an `OPENAI_API_KEY` environment variable.


In [ ]:
# !pip install -q trulens-apps-autogen trulens-providers-openai trulens-dashboard

In [ ]:
import os

from autogen import AssistantAgent
from autogen import GroupChat
from autogen import GroupChatManager
from autogen import UserProxyAgent
from trulens.apps.autogen import TruAutoGen
from trulens.core import Feedback
from trulens.core import TruSession
from trulens.core.feedback.selector import Selector
from trulens.otel.semconv.trace import SpanAttributes
from trulens.providers.openai import OpenAI as fOpenAI

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY first."

In [ ]:
session = TruSession()
session.reset_database()

## Build the group chat

The researcher can call a `search` tool; the writer turns what it finds into a
short answer. Nothing here is TruLens-specific — this is a plain AutoGen app.


In [ ]:
llm_config = {"config_list": [{"model": "gpt-4o-mini"}], "cache_seed": None}


def search(query: str) -> str:
    """Look up a topic. Stubbed here so the notebook runs without a search key."""
    facts = {
        "rayleigh scattering": (
            "Shorter wavelengths scatter more strongly off air molecules, "
            "which is why the daytime sky looks blue and sunsets look red."
        ),
    }
    for key, value in facts.items():
        if key in query.lower():
            return value
    return "No results."

In [ ]:
researcher = AssistantAgent(
    "researcher",
    llm_config=llm_config,
    system_message=(
        "You research topics. Call the search tool once, then report what it "
        "returned. Do not write the final answer yourself."
    ),
)

writer = AssistantAgent(
    "writer",
    llm_config=llm_config,
    system_message=(
        "You turn the researcher's findings into a two-sentence answer for a "
        "general audience. End your message with TERMINATE."
    ),
)

user = UserProxyAgent(
    "user",
    human_input_mode="NEVER",
    code_execution_config=False,
    max_consecutive_auto_reply=0,
)

# The researcher asks for the tool; the user proxy runs it.
researcher.register_for_llm(name="search", description="Look up a topic.")(
    search
)
user.register_for_execution(name="search")(search)

In [ ]:
group_chat = GroupChat(
    agents=[user, researcher, writer],
    messages=[],
    max_round=8,
    # `auto` lets the manager pick the next speaker, and routes a
    # tool call to the agent registered to execute it.
    speaker_selection_method="auto",
)

manager = GroupChatManager(groupchat=group_chat, llm_config=llm_config)

## Define metrics

`Selector` is what makes per-agent evaluation possible. Pointing a metric at
the `AGENT` span type and `AGENT.OUTPUT_MESSAGE` scores each agent turn on its
own, so a weak researcher shows up separately instead of being averaged into
one conversation-level number.

In [ ]:
provider = fOpenAI(model_engine="gpt-4o-mini")

f_answer_relevance = (
    Feedback(provider.relevance_with_cot_reasons, name="Answer Relevance")
    .on_input()
    .on_output()
)

f_agent_coherence = Feedback(
    provider.coherence_with_cot_reasons, name="Agent Coherence"
).on({
    "text": Selector(
        span_type=SpanAttributes.SpanType.AGENT,
        span_attribute=SpanAttributes.AGENT.OUTPUT_MESSAGE,
    ),
})

## Record the conversation

Wrapping the user proxy is enough: `TruAutoGen` instruments the AutoGen classes,
so the manager, the researcher, the writer, and the tool call are all recorded.


In [ ]:
tru_recorder = TruAutoGen(
    user,
    app_name="research_team",
    app_version="auto_selection",
    main_method=user.initiate_chat,
    feedbacks=[f_answer_relevance, f_agent_coherence],
)

In [ ]:
with tru_recorder as recording:
    result = user.initiate_chat(
        manager,
        message="Why is the sky blue?",
    )

print(result.summary)

## Inspect the span tree

Each round shows up as a speaker selection followed by that agent's reply, with
the tool call nested under the agent that ran it.


In [ ]:
session.force_flush()

events = session.get_events(
    app_name="research_team", app_version="auto_selection"
)

events.assign(
    span_type=lambda df: df["record_attributes"].apply(
        lambda a: a.get(SpanAttributes.SPAN_TYPE)
    ),
    agent=lambda df: df["record_attributes"].apply(
        lambda a: a.get(SpanAttributes.AGENT.NAME)
    ),
    reply=lambda df: df["record_attributes"].apply(
        lambda a: str(a.get(SpanAttributes.AGENT.OUTPUT_MESSAGE, ""))[:80]
    ),
)[["span_type", "agent", "reply"]]

## View results


In [ ]:
session.get_leaderboard()

## Launch the dashboard


In [ ]:
from trulens.dashboard import run_dashboard

run_dashboard()